### Testing knn accuracy of TF-IDF models

In [68]:
import numpy as np
import pandas as pd
import json
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import datasets
import src
from mteb.evaluation.evaluators.utils import get_vocab

In [55]:
def knn_accuracy(embeddings, true_labels, test_size=0.1, k = 10, rs=42, metric="euclidean"):
    """Calculates kNN accuracy.
    
    Parameters
    ----------
    embeddings : list 
        List with the different datasets for which to calculate the kNN accuracy.
    true_labels : array-like
        Array with labels (colors).
    k : int, default=10
        Number of nearest neighbors to use.
    rs : int, default=42
        Random seed.
    metric : str, default="euclidean"
        Metric to use for the distances computation (e.g. "euclidean", "cosine", etc.).
    
    Returns
    -------
    knn_accuracy : float
        kNN accuracy of the dataset.
    
    """

    X_train, X_test, y_train, y_test = train_test_split(embeddings, true_labels, test_size=test_size, random_state = rs)
    knn = KNeighborsClassifier(n_neighbors=k, algorithm='brute', n_jobs=-1, metric=metric)
    knn = knn.fit(X_train, y_train)
    knn_accuracy = knn.score(X_test, y_test)

    
    return knn_accuracy

In [16]:
# load datasets
data_batched = {
"arxiv": datasets.load_dataset("mteb/arxiv-clustering-p2p", revision="a122ad7f3f0291bf49cc6f4d32aa80929df69d5d")["test"],
"biorxiv": datasets.load_dataset("mteb/biorxiv-clustering-p2p", revision="f5dbc242e11dd8e24def4c4268607a49e02946dc")["test"],
"medrxiv": datasets.load_dataset("mteb/medrxiv-clustering-p2p", revision="e7a26af6f3ae46b30dde8737f02c07b1505bcc73")["test"],
"reddit": datasets.load_dataset("mteb/reddit-clustering-p2p", revision="385e3cb46b4cfa89021f56c4380204149d0efe33")["test"],
"stackexchange": datasets.load_dataset("mteb/stackexchange-clustering-p2p", revision="815ca46b2622cec33ccafc3735d572c266efdb44")["test"]
}

In [20]:
data_batched["arxiv"]

Dataset({
    features: ['sentences', 'labels'],
    num_rows: 31
})

In [48]:
len([split["labels"] for split in data_batched["biorxiv"]])

53787

In [51]:
data_full = {}
for name, data in data_batched.items():
    if name == "biorxiv":
        labels = [split["labels"] for split in data]
        sentences = [split["sentences"] for split in data]
    else:    
        labels = [x for split in data for x in split["labels"]]
        sentences = [x for split in data for x in split["sentences"]]
    data_full[name] = {"sentences": sentences, "labels": labels}

dataframe
colums: models
rows: datasets (full and batched)

In [15]:
d = {"model_a": {"score1": 1, "score2":2}, "model_b": {"score1": 3, "score2":4}}
pd.DataFrame(d)

,model_a,model_b
score1,1,3
score2,2,4


In [30]:
#dict for results
results_full = {}

In [ ]:
# get knn acc for full data 
def knn_acc_full(model, data_full):
    scores = {}

    for name, data in data_full.items():
        print(name)
        if model.mteb_model_meta.revision == 'tfidf_log':
            if name in ["arxiv", "reddit"]:
                continue
            embeddings = model.encode(sentences=data["sentences"])
        score = knn_accuracy(embeddings, data["labels"])
        scores[name + "_full"] = score
    
    return scores

Models to test:
- tfidf_log 
- tfidf_svd_log
- tfidf_svd_log_piecewise

In [63]:
# get knn acc for full data 
model = src.tfidf_log.Tfidf()
scores = {}

for name, data in data_full.items():
    print(name)
    if name in ["arxiv", "reddit"]:
        continue            
    embeddings = model.encode(sentences=data["sentences"])
    scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])
    

results_full[model.mteb_model_meta.revision] = scores

arxiv
biorxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(53787, 168265)
dense matrix shape(53787, 168265)
medrxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(37500, 73110)
dense matrix shape(37500, 73110)
reddit
stackexchange
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(75000, 120133)
dense matrix shape(75000, 120133)


In [80]:
results_full

{'tfidf_log': {'biorxiv_full': 0.6066183305447109,
  'medrxiv_full': 0.6466666666666666,
  'stackexchange_full': 0.47533333333333333}}

In [ ]:
with open("text_embedding/MTEB/knn_results/knn_results_2.json", "w") as fp:
    json.dump(results_full , fp, indent = 4) 

In [65]:
results_batchwise = {}

In [ ]:
# get knn acc for batched data
model = src.tfidf_log.Tfidf()
scores = {}

for name, data in data_batched.items():
    if name == "biorxiv": # biorxiv is the only one without batches
        continue
    # get full data to compute unified vocabulary
    vocab = get_vocab(data_full[name]["sentences"])
    # save scores for each batch
    batch_scores = []
    # iterate over batches
    for split in data:
        embeddings = model.encode(sentences=split["sentences"], vocab = vocab)
        batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
    scores[name + "_batchwise"] = batch_scores

results_full[model.mteb_model_meta.revision] = scores

get_vocab called!
input: <class 'list'> of length 732723
vocab of lenght 331735 starting with ['0000011d', '000026yorke', '00007h', '0001035v2', '00015t', '00019d', '0002bx', '0002d', '0002ev', '0002msun']
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 331735)
tfidf matrix shape(25000, 331735)
dense matrix

In [62]:
results_full

{'tfidf_log': {'stackexchange_full': 0.47533333333333333}}